# s06 - FedAvg vs FedProx + biaya komunikasi: R1 & R4 (SURVEY)

**Klaim R1** (§4.2): FedProx lebih stabil di non-IID dibanding FedAvg [F16][F31].
**Klaim R4** (§4.2): biaya komunikasi FL [F8] (bytes/round + latensi/throughput).

**R1 (disimulasikan di notebook):** cross-dataset {CIC,UNSW} dengan mu in {0, 0.01,
0.1}; logika FedProx IDENTIK `fed_node_survey.py` (proximal term). Bandingkan MCC
global + rounds-to-converge.

**R4 (bytes/round dihitung analitis dari ukuran bobot):** parameter model x 2 arah x
K klien x byte/param. Latensi/throughput NYATA diukur saat run EC2 (lihat runbook
`fed-runbook-survey.md` §6) dan diisi manual — notebook ini mengisi komponen bytes.

**Prasyarat:** partisi s01. **Output:** `survey_out/r1_fedprox.csv`, `r4_comm.csv`.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'; BASE='unsw-far/federated_survey'
import boto3; s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; MUS=[0.0,0.01,0.1]; SEEDS=[13,42,101]

In [ ]:
P={}
for nm in ['cic_train','cic_test','unsw_train','unsw_test']:
    sub='cic' if nm.startswith('cic') else 'unsw'
    d=load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz'); assert d is not None,'jalankan s01 dulu'
    P[nm]=(np.asarray(d['X'],float),np.asarray(d['y'],int))
Xg_te=np.vstack([P['cic_test'][0],P['unsw_test'][0]]); yg_te=np.concatenate([P['cic_test'][1],P['unsw_test'][1]])
Xtr_all=np.vstack([P['cic_train'][0],P['unsw_train'][0]])
LAYERS=[Xg_te.shape[1],128,64,1]

## 1. MLP + FedProx (proximal term identik fed_node_survey.py)

In [ ]:
def init_w(seed):
    rng=np.random.RandomState(seed); W=[]
    for a,b in zip(LAYERS[:-1],LAYERS[1:]): W.append(rng.randn(a,b)*np.sqrt(2/a)); W.append(np.zeros(b))
    return W
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs,lr,mu,Wg,seed):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,256):
            bi=idx[s:s+256]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2)=fwd(W,Xb); g=(o-yb)/max(len(yb),1)
            gW4=z2.T@g[:,None]; gb4=g.sum(0); d2=(g[:,None]@W[4].T)*(a2>0)
            gW2=z1.T@d2; gb2=d2.sum(0); d1=(d2@W[2].T)*(a1>0); gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0: gv=gv+mu*(W[i]-Wg[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(Ws,ns): tot=sum(ns); return [sum((ns[k]/tot)*Ws[k][i] for k in range(len(Ws))) for i in range(len(Ws[0]))]
def zfit(X): m=X.mean(0); s=X.std(0); s[s==0]=1; return m,s
m_mon,s_mon=zfit(Xtr_all)
clients={'cic':(( P['cic_train'][0]-m_mon)/s_mon, P['cic_train'][1]),
         'unsw':((P['unsw_train'][0]-m_mon)/s_mon,P['unsw_train'][1])}
Xg_te_z=(Xg_te-m_mon)/s_mon

## 2. R1: sweep mu x seed; catat kurva MCC utk rounds-to-converge

In [ ]:
def run(mu,seed,track=False):
    W=init_w(seed); curve=[]
    for t in range(ROUNDS):
        Ws=[];ns=[]
        for cid,(Xc,yc) in clients.items():
            Ws.append(train_local(W,Xc,yc,LOCAL_EPOCHS,LR,mu,W,seed)); ns.append(len(yc))
        W=fedavg(Ws,ns)
        if track:
            o,_=fwd(W,Xg_te_z); curve.append(matthews_corrcoef(yg_te,(o>=0.5).astype(int)))
    o,_=fwd(W,Xg_te_z); return matthews_corrcoef(yg_te,(o>=0.5).astype(int)),curve
def rounds_to_converge(curve,tol=0.01):
    if not curve: return None
    final=curve[-1]
    for t,v in enumerate(curve):
        if abs(v-final)<=tol: return t
    return len(curve)
rows=[]
for mu in MUS:
    finals=[]; r2c=[]
    for sd in SEEDS:
        f,curve=run(mu,sd,track=(sd==SEEDS[0])); finals.append(f)
        if curve: r2c.append(rounds_to_converge(curve))
    rows.append(dict(method='FedProx' if mu>0 else 'FedAvg', mu=mu,
                     mcc_mean=float(np.mean(finals)), mcc_std=float(np.std(finals)),
                     rounds_to_converge=int(np.mean(r2c)) if r2c else -1))
    print(f'mu={mu:<5} MCC={rows[-1]["mcc_mean"]:.3f}+/-{rows[-1]["mcc_std"]:.3f} r2c={rows[-1]["rounds_to_converge"]}')

## 3. R4: komponen bytes/round (analitis). Latensi/throughput diisi dari log EC2.

In [ ]:
n_params=sum(w.size for w in init_w(0)); K=len(clients); bytes_per_param=8  # float64
bytes_up=n_params*bytes_per_param               # client -> server, per client per round
bytes_round=bytes_up*K*2                          # 2 arah (download global + upload update)
comm=dict(n_params=n_params,K=K,bytes_per_update=bytes_up,bytes_per_round=bytes_round,
          kb_per_round=round(bytes_round/1024,1),
          latency_s_per_round='[TBD dari log EC2]',throughput='[TBD dari log EC2]')
print(comm)

In [ ]:
import csv
p1=os.path.join(OUTDIR,'r1_fedprox.csv')
with open(p1,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['method','mu','mcc_mean','mcc_std','rounds_to_converge']); w.writeheader()
    for r in rows: w.writerow(r)
p2=os.path.join(OUTDIR,'r4_comm.csv')
with open(p2,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(comm.keys())); w.writeheader(); w.writerow(comm)
print('saved',p1,p2)
for p,key in [(p1,f'{BASE}/results/r1_fedprox.csv'),(p2,f'{BASE}/results/r4_comm.csv')]:
    try: s3.upload_file(p,BUCKET,key)
    except Exception as e: print('(upload',p,'dilewati:',e,')')
print('=== s06 SELESAI. Isi §4.3 baris R1 (dari r1_fedprox) & R4 (bytes dari r4_comm; latensi dari log EC2). ===')